In [1]:
!pip install cryptography

In [2]:
import hashlib
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization

class Wallet:
    def __init__(self, name):
        self.name = name
        # 1. Generate Keys
        self.private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        self.public_key = self.private_key.public_key()
        # 2. Derive Address (Hash of Public Key)
        pub_bytes = self.public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
        self.address = hashlib.sha256(pub_bytes).hexdigest()[:16]
        self.balance = 100  # Starting balance for simulation

    def sign_transaction(self, message):
        # 3. Create Signature
        signature = self.private_key.sign(
            message.encode(),
            padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
            hashes.SHA256()
        )
        return signature

class BlockchainSystem:
    def __init__(self):
        self.processed_transaction_ids = set() # To prevent double-spending

    def verify_and_process(self, sender, receiver, amount, tx_id, signature):
        # 4. Check Double-Spending
        if tx_id in self.processed_transaction_ids:
            return "REJECTED: Double-spending detected (TX ID already exists)!"

        # 5. Check Balance
        if sender.balance < amount:
            return "REJECTED: Insufficient funds!"

        # 6. Verify Signature
        try:
            message = f"{sender.address}{receiver.address}{amount}{tx_id}"
            sender.public_key.verify(
                signature,
                message.encode(),
                padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
                hashes.SHA256()
            )

            # If verification passes, update balances
            sender.balance -= amount
            receiver.balance += amount
            self.processed_transaction_ids.add(tx_id)
            return "SUCCESS: Transaction processed and added to ledger."

        except Exception:
            return "REJECTED: Invalid Signature (Tampering detected)!"

# --- DEMONSTRATION ---

# Initialize Wallets and System
network = BlockchainSystem()
wallet_a = Wallet("A")
wallet_b = Wallet("B")

def create_tx(sender, receiver, amount, custom_id=None):
    tx_id = custom_id or hashlib.sha256(f"{sender.address}{receiver.address}{amount}".encode()).hexdigest()
    msg = f"{sender.address}{receiver.address}{amount}{tx_id}"
    sig = sender.sign_transaction(msg)
    return sender, receiver, amount, tx_id, sig

# Test 1: Valid Transaction
print("Test 1: Sending 40 coins from A to B...")
tx1 = create_tx(wallet_a, wallet_b, 40)
print(network.verify_and_process(*tx1))

# Test 2: Double-Spending Attempt (using same tx1 data)
print("\nTest 2: Attempting to reuse the same transaction ID...")
print(network.verify_and_process(*tx1))

# Test 3: Tampering Attempt (Changing amount after signing)
print("\nTest 3: Attempting to change amount to 100 without resigning...")
sender, receiver, amount, tx_id, sig = create_tx(wallet_a, wallet_b, 10)
print(network.verify_and_process(sender, receiver, 100, tx_id, sig)) # Modified amount

Test 1: Sending 40 coins from A to B...
SUCCESS: Transaction processed and added to ledger.

Test 2: Attempting to reuse the same transaction ID...
REJECTED: Double-spending detected (TX ID already exists)!

Test 3: Attempting to change amount to 100 without resigning...
REJECTED: Insufficient funds!
